# Eksperimen Machine Learning dengan PyCaret

Notebook ini menjalankan pipeline klasifikasi sentimen menggunakan PyCaret.
Data yang digunakan adalah **output dari `02_preprocessing.ipynb`** — tidak melakukan preprocessing ulang.

**Urutan eksekusi yang wajib diikuti:**
1. ✅ Jalankan `02_preprocessing.ipynb` terlebih dahulu
2. ▶ Jalankan notebook ini

**Output notebook ini:**
- `../models/model_ml/sentiment_model.pkl` — model PyCaret final
- `../models/model_ml/pipeline.pkl` — sudah tersimpan dari notebook 02


## 1. Import Library


In [ ]:
import os, pickle, warnings, re
from pathlib import Path
from typing import List, Optional, Union
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer as _SklearnTFIDF
import logging

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.WARNING)
logger = logging.getLogger(__name__)

from pycaret.classification import (
    setup, compare_models, create_model, tune_model,
    evaluate_model, predict_model, finalize_model, save_model, pull
)
print('✅ Import selesai')

## 2. Definisi Class (Inline)

> **Mengapa diperlukan?** `pipeline.pkl` disimpan ketika class `TextPreprocessor` dll. terdaftar di namespace `__main__` (notebook 02). Agar pickle bisa memuatnya di notebook ini, class-class tersebut harus didefinisikan ulang di sini.


In [ ]:
class TextPreprocessor:
    def __init__(self, lowercase=True, remove_url=True, remove_mention_hashtag=True,
                 remove_punctuation=True, remove_numbers=True, min_token_length=2,
                 slang_dict=None):
        self.lowercase = lowercase
        self.remove_url = remove_url
        self.remove_mention_hashtag = remove_mention_hashtag
        self.remove_punctuation = remove_punctuation
        self.remove_numbers = remove_numbers
        self.min_token_length = min_token_length
        self._url_re        = re.compile(r'https?://\S+|www\.\S+')
        self._mention_re    = re.compile(r'@\w+')
        self._hashtag_re    = re.compile(r'#\w+')
        self._punct_re      = re.compile(r'[^\w\s]')
        self._number_re     = re.compile(r'\d+')
        self._whitespace_re = re.compile(r'\s+')
        self.slang_mapping = {}
        if isinstance(slang_dict, dict):
            self.slang_mapping = slang_dict
        elif slang_dict is not None:
            self._load_slang_csv(slang_dict)

    def _load_slang_csv(self, path):
        path = Path(path)
        if not path.exists(): return
        try:
            df = pd.read_csv(path)
            if 'slang' in df.columns and 'formal' in df.columns:
                df = df.dropna(subset=['formal'])
                self.slang_mapping = dict(zip(df['slang'], df['formal']))
        except Exception: pass

    def clean(self, text):
        if not isinstance(text, str): text = str(text)
        if self.lowercase: text = text.lower()
        if self.remove_url: text = self._url_re.sub(' ', text)
        if self.remove_mention_hashtag:
            text = self._mention_re.sub(' ', text)
            text = self._hashtag_re.sub(' ', text)
        if self.remove_punctuation: text = self._punct_re.sub(' ', text)
        if self.remove_numbers: text = self._number_re.sub('', text)
        tokens = self._whitespace_re.split(text.strip())
        return ' '.join(str(self.slang_mapping.get(t, t)) for t in tokens if len(t) >= self.min_token_length)

    def transform(self, texts):
        return [self.clean(t) for t in texts]

class TFIDFVectorizer:
    def __init__(self, max_features=5000, ngram_range=(1,2), min_df=3, max_df=0.90, sublinear_tf=True):
        self.max_features = max_features
        self.ngram_range  = ngram_range
        self.min_df       = min_df
        self.max_df       = max_df
        self.sublinear_tf = sublinear_tf
        self._is_fitted   = False
        self._vectorizer  = _SklearnTFIDF(
            max_features=max_features, ngram_range=ngram_range,
            min_df=min_df, max_df=max_df, sublinear_tf=sublinear_tf, analyzer='word')

    def fit(self, texts):
        self._vectorizer.fit(texts); self._is_fitted = True; return self

    def transform(self, texts):
        if not self._is_fitted: raise RuntimeError('TFIDFVectorizer belum di-fit.')
        m = self._vectorizer.transform(texts)
        return pd.DataFrame(m.toarray(), columns=self._vectorizer.get_feature_names_out())

    def fit_transform(self, texts):
        return self.fit(texts).transform(texts)

    @property
    def feature_names(self): return list(self._vectorizer.get_feature_names_out())
    @property
    def n_features(self): return len(self.feature_names)

class PreprocessingPipeline:
    def __init__(self, text_col='review_text', target_col='sentiment',
                 preprocessor=None, vectorizer=None, slang_dict=None):
        self.text_col     = text_col
        self.target_col   = target_col
        self.preprocessor = preprocessor or TextPreprocessor(slang_dict=slang_dict)
        self.vectorizer   = vectorizer   or TFIDFVectorizer()
        self._is_fitted   = False

    def fit_transform(self, source, save_cleaned=False, cleaned_path='data/processed/cleaned_text.csv'):
        df = pd.read_csv(source) if not isinstance(source, pd.DataFrame) else source.copy()
        texts = df[self.text_col]; labels = df[self.target_col]
        cleaned = self.preprocessor.transform(texts)
        if save_cleaned:
            Path(cleaned_path).parent.mkdir(parents=True, exist_ok=True)
            pd.DataFrame({'clean_text': cleaned, self.target_col: labels}).to_csv(cleaned_path, index=False)
        X_df = self.vectorizer.fit_transform(cleaned)
        X_df[self.target_col] = labels.reset_index(drop=True)
        self._is_fitted = True
        return X_df

    def transform_raw(self, texts):
        if isinstance(texts, str): texts = [texts]
        return self.vectorizer.transform(self.preprocessor.transform(texts))

    def save(self, path='pipeline.pkl'):
        Path(path).parent.mkdir(parents=True, exist_ok=True)
        with open(path, 'wb') as f: pickle.dump(self, f)

    @property
    def n_features(self): return self.vectorizer.n_features

print('✅ Class definitions selesai')

In [ ]:
class _NotebookUnpickler(pickle.Unpickler):
    _REMAP = {
        'TextPreprocessor'     : TextPreprocessor,
        'TFIDFVectorizer'      : TFIDFVectorizer,
        'PreprocessingPipeline': PreprocessingPipeline,
    }
    def find_class(self, module, name):
        if name in self._REMAP: return self._REMAP[name]
        return super().find_class(module, name)

print('✅ Custom unpickler siap')

## 3. Load Data & Pipeline


In [ ]:
df_clean = pd.read_csv('../data/processed/cleaned_text.csv')

# Handle NaN di kolom clean_text
n_nan = df_clean['clean_text'].isna().sum()
if n_nan > 0:
    print(f'⚠️  {n_nan} baris NaN ditemukan di clean_text → akan diisi string kosong')
df_clean['clean_text'] = df_clean['clean_text'].fillna('').astype(str)

print(f'✅ Data dimuat: {df_clean.shape}')
print(f'   Kolom     : {list(df_clean.columns)}')
print(f'   Distribusi label:\n{df_clean["sentiment"].value_counts().to_string()}')
df_clean.head(3)

In [ ]:
with open('../models/model_ml/pipeline.pkl', 'rb') as f:
    pipeline = _NotebookUnpickler(f).load()

print(f'✅ Pipeline dimuat | Fitur TF-IDF: {pipeline.n_features}')

## 4. Generate Fitur TF-IDF

Pipeline yang sudah di-fit digunakan untuk mengubah `clean_text` → matriks TF-IDF numerik yang dibutuhkan PyCaret.


In [ ]:
# clean_text sudah di-fillna('') saat load data → aman untuk transform
X_tfidf = pipeline.vectorizer.transform(df_clean['clean_text'])
X_tfidf['sentiment'] = df_clean['sentiment'].values

print(f'✅ TF-IDF features generated')
print(f'   Shape  : {X_tfidf.shape}')
print(f'   Sampel fitur: {list(X_tfidf.columns[:5])}')
X_tfidf.head(2)

## 5. Inisialisasi Environment PyCaret (`setup`)

- Train/test split: **80%/20%**
- Cross-validation: **10-Fold Stratified K-Fold**
- Normalisasi fitur: **MinMax**


In [ ]:
clf_setup = setup(
    data=X_tfidf,
    target='sentiment',
    train_size=0.8,
    fold=10,
    fold_strategy='stratifiedkfold',
    session_id=42,
    normalize=True,
    normalize_method='minmax',
    fix_imbalance=False,
    verbose=True,
    html=False
)

## 6. Bandingkan Model (`compare_models`)

Kandidat: Logistic Regression, LightGBM, SVM Linear, Naive Bayes, Random Forest


In [ ]:
best_models = compare_models(
    include=['lr', 'lightgbm', 'svm', 'nb', 'rf'],
    sort='Accuracy',
    n_select=3,
    fold=10,
    verbose=True
)
if not isinstance(best_models, list): best_models = [best_models]
print(f'\n✅ Top-3: {[type(m).__name__ for m in best_models]}')

## 7. Buat Model Terbaik (`create_model`)


In [ ]:
model = create_model(best_models[0], fold=10, round=4, return_train_score=True)

## 8. Tuning Hyperparameter (`tune_model`)


In [ ]:
tuned_model = tune_model(
    model, optimize='Accuracy', n_iter=20,
    search_library='scikit-learn', search_algorithm='random',
    fold=10, verbose=True, return_train_score=True
)

## 9. Evaluasi Visual (`evaluate_model`)


In [ ]:
evaluate_model(tuned_model)

## 10. Prediksi Holdout Set


In [ ]:
predictions = predict_model(tuned_model, verbose=True)
print('\n📊 Skor Holdout:')
print(pull().to_string())

## 11. Finalisasi & Simpan Model


In [ ]:
os.makedirs('../models/model_ml', exist_ok=True)
final_model = finalize_model(tuned_model)
save_model(final_model, '../models/model_ml/sentiment_model')

print('\n✅ Semua artefak tersimpan:')
print('   Model PyCaret : ../models/model_ml/sentiment_model.pkl')
print('   Pipeline pkl  : ../models/model_ml/pipeline.pkl')